In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from torch.utils.data import DataLoader
import torch.utils.data as data
import torch.nn as nn
import sys
from itertools import product
import warnings

!rm -r /root/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git clone https://github.com/KirillVishnyakov/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git -C /root/Architectural-Biases-in-Time-Series-Anomaly-Detection log --oneline -1

rm: cannot remove '/Architectural-Biases-in-Time-Series-Anomaly-Detection': No such file or directory
Cloning into 'Architectural-Biases-in-Time-Series-Anomaly-Detection'...
remote: Enumerating objects: 376, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 376 (delta 10), reused 15 (delta 6), pack-reused 356 (from 1)
Receiving objects: 100% (376/376), 81.83 MiB | 23.40 MiB/s, done.
Resolving deltas: 100% (196/196), done.
fatal: cannot change to '/Architectural-Biases-in-Time-Series-Anomaly-Detection': No such file or directory


In [2]:
sys.path.append("/root/Architectural-Biases-in-Time-Series-Anomaly-Detection")
import utils.config as config
config.init("/mnt/data/data/data.csv", "/root/Architectural-Biases-in-Time-Series-Anomaly-Detection/checkpoint_dir")

from models.lstm_forecaster import lstm_forecaster
from models.lstm_ae import lstm_ae
import models.transformer_encoder_forecaster as transformer_encoder_forecaster
import dataset
import train
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(device)
warnings.filterwarnings("ignore")

2.8.0+cu129
cuda


In [3]:
seed = 432
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
train_dataset_t = dataset.forecasting_Dataset(device = device, window_size = 256, horizon = 4, start = 0, end = 800000)
test_dataset_t = dataset.forecasting_Dataset(device = device, window_size = 256, horizon = 4, scaler = train_dataset_t.scaler, start = 800000, end = 1000000, train = False)

In [6]:
transformer_model = transformer_encoder_forecaster.patch_transformer(lookback_window = 256, forecast_horizon = 4, d_model = 256, nhead = 8, dropout = 0.0, num_features = 17, num_blocks = 1).to(device)
transformer_model = torch.compile(transformer_model)
best_model_wts, best_loss, avg_train_loss = \
train.fit_forecaster(device, transformer_model, "test_t", train_dataset_t, test_dataset_t, lr = 0.001, batch_size = 512, num_epochs = 20)

32
LR Scheduler: 781 warmup steps, 31240 total steps
|test_t| train = 0.1860 | test= 0.0220 | LR: 9.98e-04
|test_t| train = 0.0220 | test= 0.0210 | LR: 9.86e-04
|test_t| train = 0.0215 | test= 0.0210 | LR: 9.60e-04
|test_t| train = 0.0212 | test= 0.0209 | LR: 9.23e-04
|test_t| train = 0.0212 | test= 0.0207 | LR: 8.76e-04
|test_t| train = 0.0211 | test= 0.0206 | LR: 8.18e-04
|test_t| train = 0.0210 | test= 0.0204 | LR: 7.53e-04
|test_t| train = 0.0209 | test= 0.0204 | LR: 6.81e-04
|test_t| train = 0.0208 | test= 0.0204 | LR: 6.04e-04
|test_t| train = 0.0208 | test= 0.0204 | LR: 5.25e-04
|test_t| train = 0.0207 | test= 0.0203 | LR: 4.45e-04
|test_t| train = 0.0207 | test= 0.0203 | LR: 3.67e-04
|test_t| train = 0.0206 | test= 0.0203 | LR: 2.93e-04
|test_t| train = 0.0206 | test= 0.0202 | LR: 2.24e-04
|test_t| train = 0.0205 | test= 0.0202 | LR: 1.62e-04
|test_t| train = 0.0205 | test= 0.0202 | LR: 1.09e-04
|test_t| train = 0.0205 | test= 0.0202 | LR: 6.67e-05
|test_t| train = 0.0204 | tes

In [7]:
torch.save(transformer_model._orig_mod.state_dict(), "tranformer_forecaster_weights.pt")